# 数据资产构建与 Hugging Face Datasets

> **本章定位**：建立语言模型数据管线所需的源资产、Schema、质量字段、数据划分与持久化契约。

> **章节边界**：本章聚焦单机可复核的数据资产闭环；大规模采集、清洗、去重、配比与增量处理见 `A20_data_engineering.ipynb`，模型结构与训练目标不在本章展开。

> **总览**：内容从确定性句对构造出发，经由记录校验与 JSONL 原子发布，迁移到 Hugging Face Datasets 的类型化加载、质量变换、稳定划分、持久化和流式读取接口。

```mermaid
flowchart LR
    A["确定性句对"] --> B["记录校验"]
    B --> C["原子发布 JSONL"]
    C --> D["load_dataset + Features"]
    D --> E["map / filter"]
    E --> F["DatasetDict"]
    F --> G["save_to_disk / load_from_disk"]
    D --> H["streaming=True"]
```


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 共同基础：数据契约；跨方向数据专题 |
| 本章定位 | 建立可审计的数据源、质量校验、稳定划分、持久化与流式读取契约。 |
| 先修知识 | 完成 `10_foundations.ipynb`；能够使用 Python 列表、字典、文件与函数。 |
| 预计时间 | 60～90 分钟 |
| 运行资源 | CPU 即可；本地磁盘需要少量临时空间。 |
| 输入 | 按稳定规则生成的中英平行句对。 |
| 交付物 | JSONL 权威源资产、类型化 `DatasetDict`、划分成员记录与内容哈希。 |

### 1.1．学习目标

完成本章后，读者能够构造并原子发布 JSONL 权威源资产，使用显式 Schema 完成类型化加载和质量变换，建立与泄漏单位一致的数据划分，并通过内容哈希与重载检查证明数据版本的完整性和可恢复性。


## 2．直觉与输入输出契约

### 2.1．数据划分与泄漏控制

训练集用于拟合模型参数以及词表、标准化器、去重阈值等数据相关状态；验证集用于选择超参数、阈值和 Checkpoint；测试集仅在方案冻结后提供最终估计。测试集一旦参与反复选择，就失去独立评估作用。

划分单位应与潜在泄漏单位一致。同一文档、用户、会话或近重复样本需要整体进入同一 split；具有类别标签的数据可采用分层划分；预测未来的任务按时间先后划分。随机逐行划分只适用于样本彼此独立且不存在实体或时间泄漏的场景。

### 2.2．输入输出契约

| 阶段 | 输入 | 输出 | 关键约束 |
|---|---|---|---|
| 规则构造 | 主语、宾语和句式模板 | `list[tuple[str, str]]`，共 72 条 | 顺序稳定，文本非空 |
| JSONL 发布 | `list[dict]` | UTF-8 `data.jsonl`，每行一条记录 | 字段为 `zh` 与 `en`，写入完成后原子替换 |
| 类型化加载 | JSONL 与 `Features` | 72 行 Arrow `Dataset` | 两列均为字符串 |
| 质量变换 | `Dataset` | 增加 `row_id`、内容哈希和长度字段的数据集 | 原始文本保持不变，哈希用于精确去重 |
| 稳定划分 | 质量数据集与划分配置 | `DatasetDict`：`58/7/7` | split 之间的内容哈希不重叠 |
| 持久化 | `DatasetDict` | 可重载的 Arrow 目录 | 重载后的 Schema、行数与成员顺序一致 |

<!-- diagram:dataset-asset-splits -->

![架构图：从权威 JSONL 到稳定数据划分的数据资产拓扑](assets/figures/20_dataset/dataset-asset-splits.svg)

[TikZ 源文件](assets/figures/20_dataset/dataset-asset-splits.tex)


In [ ]:
# 原理实现仅使用 Python 标准库；Datasets 在库迁移章节按需导入。
import hashlib
import json
from pathlib import Path


def my_locate_project_directory():
    """定位 Notebook 工作区中的 llm_from_scratch 目录；目录不存在时回退到当前目录。"""
    current_directory = Path.cwd()
    project_directory = current_directory / "llm_from_scratch"
    # Jupyter 通常从 Notebook 目录启动；Colab 则从 /content 启动且不暴露 ipynb 文件。
    # 两种环境都应可写，因此不再把 Notebook 文件是否可见作为运行前提。
    return project_directory if project_directory.is_dir() else current_directory


PROJECT_DIRECTORY = my_locate_project_directory()


<!-- theory-math-contract:v1 -->
### 2.3．核心机制的语言与数学表达

数据划分的核心不是“按比例切几份”，而是让同一原子样本或同一泄漏组稳定地只属于一个集合。可将规范化后的分组键 $g(x)$ 哈希到 $M$ 个桶：

$$
b(x)=\operatorname{int}\!\left(H(g(x))[:k],16\right)\bmod M,
\qquad
D_{\mathrm{train}}\cap D_{\mathrm{valid}}=D_{\mathrm{train}}\cap D_{\mathrm{test}}=\varnothing
$$

其中，$H$ 是固定版本的内容哈希，$g(x)$ 是样本、会话、文档或问题家族的稳定分组键，$b(x)$ 是桶编号，$M$ 是桶总数。`stable_split` 对应 $b(x)$，`DatasetDict` 承载三个互斥集合。公式保证的是给定规范化、分组键和哈希算法下的确定性；如果清洗规则或分组粒度变化，划分成员也会变化，必须生成新的数据版本。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．确定性构造候选句对

源语料覆盖陈述句、进行时、问句和请求句。笛卡尔积与固定补充列表按稳定顺序生成全部候选句对，本阶段不建立训练、验证或测试划分。


In [ ]:
# 按固定模板确定性生成中英句对，保证每次运行得到相同源数据。

def my_build_parallel_pairs():
    """通过固定模板确定性构建中英平行句对列表。"""
    pairs = []

    def my_add_grid(subjects, objects, make_pair):
        """对主语和宾语做笛卡尔积，并把模板生成的句对追加到外层 pairs 列表。"""
        # 对主语和宾语做笛卡尔积，再由模板生成一组平行句对。
        for subject in subjects:
            for object_item in objects:
                pairs.append(make_pair(subject, object_item))

    like_subjects = [("我", "i"), ("你", "you"), ("我们", "we"), ("他们", "they")]
    things = [
        ("苹果", "apples"), ("香蕉", "bananas"), ("音乐", "music"),
        ("咖啡", "coffee"), ("中文", "chinese"), ("数学", "mathematics"),
        ("编程", "programming"), ("人工智能", "artificial intelligence"),
    ]
    my_add_grid(
        like_subjects,
        things,
        lambda subject, thing: (
            f"{subject[0]}喜欢{thing[0]}。",
            f"{subject[1]} like {thing[1]}.",
        ),
    )

    learning_subjects = [
        ("我", "i am"), ("你", "you are"),
        ("我们", "we are"), ("他们", "they are"),
    ]
    topics = [
        ("中文", "chinese"), ("英文", "english"), ("数学", "mathematics"),
        ("编程", "programming"), ("人工智能", "artificial intelligence"),
        ("机器学习", "machine learning"),
    ]
    my_add_grid(
        learning_subjects,
        topics,
        lambda subject, topic: (
            f"{subject[0]}正在学习{topic[0]}。",
            f"{subject[1]} learning {topic[1]}.",
        ),
    )

    # 在规则网格之外补充问句和指令句，增加句式覆盖。
    pairs.extend(
        (f"你喜欢{zh}吗？", f"do you like {en}?")
        for zh, en in things
    )
    pairs.extend(
        [
            ("请打开门。", "please open the door."),
            ("请关闭窗户。", "please close the window."),
            ("请阅读这本书。", "please read this book."),
            ("请播放音乐。", "please play music."),
            ("请学习中文。", "please study chinese."),
            ("请打开这本书。", "please open this book."),
            ("请关闭门。", "please close the door."),
            ("请打开窗户。", "please open the window."),
        ]
    )
    return pairs


ALL_PAIRS = my_build_parallel_pairs()


### 3.2．源记录校验

期望行数由生成规则推导为 72。代码核对实际行数并输出首尾记录，使模板缺失、重复追加或顺序漂移能够被及时识别。非空校验、内容哈希与 split 隔离在后续类型化处理阶段继续完成。


In [ ]:
# 将候选句对整理为 records，作为后续 JSONL 与 Dataset 的唯一数据源。

EXPECTED_ROW_COUNT = 72  # 生成规则的数据契约值；模板或候选集合变化时必须同步推导并保留校验。
if len(ALL_PAIRS) != EXPECTED_ROW_COUNT:
    raise RuntimeError(
        f"句对生成规则应产生 {EXPECTED_ROW_COUNT} 条，实际得到 {len(ALL_PAIRS)} 条"
    )

print("完整数据行数：", len(ALL_PAIRS))
print("首条记录：", ALL_PAIRS[0])
print("末条记录：", ALL_PAIRS[-1])


### 3.3．JSONL 序列化与原子发布

全部 72 条记录序列化为 UTF-8 `data.jsonl`，每行仅包含中英文文本：

```json
{"zh":"我喜欢苹果。","en":"i like apples."}
```

序列化固定字段顺序与紧凑分隔符，以获得稳定内容哈希。文件先写入同目录临时路径，再通过原子替换发布；随后重新读取每一行，形成明确的持久化边界。


In [ ]:
# 先写入临时文件再原子替换，避免中断时留下不完整 JSONL。

DATASET_DIRECTORY_NAME = "translation_parallel"
DATA_FILENAME = "data.jsonl"

records = [
    {"zh": chinese, "en": english}
    for chinese, english in ALL_PAIRS
]
data_payload = (
    "\n".join(
        json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
        for record in records
    )
    + "\n"
).encode("utf-8")

DATASET_DIRECTORY = PROJECT_DIRECTORY / "assets" / "data" / DATASET_DIRECTORY_NAME
DATASET_DIRECTORY.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATASET_DIRECTORY / DATA_FILENAME
temporary_path = DATA_PATH.with_suffix(DATA_PATH.suffix + ".tmp")
temporary_path.write_bytes(data_payload)
temporary_path.replace(DATA_PATH)

reloaded_payload = DATA_PATH.read_bytes()
reloaded_records = [json.loads(line) for line in reloaded_payload.splitlines()]

print("完整 JSONL 数据集已原子写入并重新加载")
print("  data：", DATA_PATH.relative_to(PROJECT_DIRECTORY))
print("  rows：", len(records))
print("  sha256：", hashlib.sha256(data_payload).hexdigest())


## 4．证据验证

### 4.1．记录、Schema 与内容哈希

标准库路径提供行数、首尾记录、发布路径和完整 SHA-256。Hugging Face `Dataset.from_list` 随后以显式 `Features` 读取同一 `records`，用于核对以下证据：

- 原理实现与类型化数据集均包含 72 行。
- 列名固定为 `zh`、`en`，两列类型均为 `Value("string")`。
- JSONL 重新加载后的每条记录与内存记录保持相同字段和值。
- 相同生成规则与序列化配置产生相同内容哈希。

`Dataset.from_list` 的输出由 Apache Arrow 承载。显式 `Features` 将字段名称和类型转化为可检查契约，避免跨文件或脏数据场景中的意外类型推断。依赖缺失时使用项目统一环境安装方式补齐，并以代码输出的 `datasets.__version__` 记录实际版本。


In [ ]:
# 为 Dataset 显式声明 Features，固定字段名称和字符串类型。

import datasets
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk

translation_features = Features({
    "zh": Value("string"),
    "en": Value("string"),
})
dataset_from_records = Dataset.from_list(
    records,
    features=translation_features,
)


print("datasets:", datasets.__version__)
print(dataset_from_records)
print(dataset_from_records.features)


## 5．迁移到生产库

### 5.1．使用 `load_dataset` 加载权威源资产

`Dataset.from_list` 适用于已有 Python records 的场景；文件、对象存储与 Hub 数据通常通过 `load_dataset` 进入数据管线。本节以同一 JSONL 为唯一源资产，显式命名为 `full` split，并复用 `translation_features` 构建类型化 Arrow 数据集和缓存。

输入是 `data.jsonl` 与字段契约，输出是包含 `full` 的 `DatasetDict`。迁移验收以行数、字段、类型与逐行内容一致为准。

参考：[Hugging Face Datasets：Load](https://huggingface.co/docs/datasets/loading)。


In [ ]:
# 通过 load_dataset 重新读取 JSONL，切换到 Hugging Face 标准数据接口。

loaded_dataset_dict = load_dataset(
    "json",
    data_files={"full": str(DATA_PATH)},
    features=translation_features,
)
full_dataset = loaded_dataset_dict["full"]

print(loaded_dataset_dict)


### 5.2．使用 `map` 与 `filter` 建立可追踪变换

Datasets 的变换返回新对象，不会原地修改源数据。批量 `map` 增加稳定行号、内容哈希和长度特征，`filter` 执行非空质量门禁。内容哈希可用于跨 split 的精确去重，但不能替代来源、许可证和语义分组元数据。

输入为 `full_dataset`；输出增加 `row_id`、`pair_sha256`、`zh_chars`、`en_words` 四列。验收条件是行数仍为 72、内容哈希唯一且原始文本列保持不变。

参考：[Hugging Face Datasets：Process](https://huggingface.co/docs/datasets/process)。

<!-- diagram:dataset-transform-layer -->
```mermaid
flowchart LR
    J["data.jsonl<br/>权威源资产"] --> L["load_dataset + Features"]
    L --> A["Arrow Dataset"]
    A --> M["map<br/>派生质量字段"]
    M --> F["filter<br/>质量门禁"]
    F --> D["DatasetDict<br/>train / validation / test"]
    D --> B["DataLoader 或训练任务"]
```


In [ ]:
# 使用批量 map 增加质量字段，再用 filter 保留符合规则的记录。

def my_add_quality_columns(batch, indices):
    """为批量中英句对生成行号、内容摘要及中英文长度质量字段。"""
    return {
        "row_id": indices,
        "pair_sha256": [
            hashlib.sha256(f"{zh}␟{en}".encode("utf-8")).hexdigest()
            for zh, en in zip(batch["zh"], batch["en"])
        ],
        "zh_chars": [len(text) for text in batch["zh"]],
        "en_words": [len(text.split()) for text in batch["en"]],
    }


quality_dataset = full_dataset.map(
    my_add_quality_columns,
    batched=True,
    with_indices=True,
    desc="构建质量字段",
)
quality_dataset = quality_dataset.filter(
    lambda row: bool(row["zh"].strip() and row["en"].strip()),
    desc="过滤空句对",
)

print(quality_dataset)
print(quality_dataset[0])


### 5.3．确定性生成 `DatasetDict`

固定 seed 与整数样本数形成 `58/7/7` 的 `train`、`validation`、`test`。整数规模避免小数据集的浮点比例取整歧义；相同源数据、Datasets 版本与划分配置应产生相同成员和顺序。

代码同时记录每个 split 的 `pair_sha256` 集合和期望规模，为成员重叠检查及后续 Manifest 提供数据。


In [ ]:
# 按固定随机种子生成 DatasetDict，使数据划分可重复。

SPLIT_SEED = 42  # 固定成员归属与顺序；修改后必须登记新的数据版本。
HOLDOUT_ROW_COUNT = 14  # 72 条中的留出规模，仅用于管线验证；正式比例需按泄漏单位与统计功效确定。
TEST_ROW_COUNT_WITHIN_HOLDOUT = 7  # 将留出集等分为 7 条验证与 7 条测试记录。
train_and_holdout = quality_dataset.train_test_split(
    test_size=HOLDOUT_ROW_COUNT,
    seed=SPLIT_SEED,
    shuffle=True,
)
validation_and_test = train_and_holdout["test"].train_test_split(
    test_size=TEST_ROW_COUNT_WITHIN_HOLDOUT,
    seed=SPLIT_SEED,
    shuffle=True,
)
dataset_dict = DatasetDict({
    # DatasetDict 使用主流库约定的三个 split 名称。
    "train": train_and_holdout["train"],
    "validation": validation_and_test["train"],
    "test": validation_and_test["test"],
})

expected_split_sizes = {"train": 58, "validation": 7, "test": 7}  # 当前 72 条数据的验收契约。
actual_split_sizes = {name: len(split) for name, split in dataset_dict.items()}
if actual_split_sizes != expected_split_sizes:
    raise RuntimeError(
        f"split 大小不符合数据契约：expected={expected_split_sizes}, actual={actual_split_sizes}"
    )

split_hashes = {
    name: set(split["pair_sha256"])
    for name, split in dataset_dict.items()
}

repeated_split = quality_dataset.train_test_split(
    test_size=HOLDOUT_ROW_COUNT,
    seed=SPLIT_SEED,
    shuffle=True,
)
print(dataset_dict)


#### 5.3.1．划分规模与成员隔离的可视化证据

学习问题是：`58/7/7` 的规模契约与内容哈希隔离是否同时成立。左图直接读取 `DatasetDict` 的实际行数；右图计算任意两个 Split 的 `pair_sha256` 交集。验收条件是对角线等于各 Split 行数，所有非对角线均为 0。


In [ ]:
# 使用真实 Split 成员绘制规模与哈希交集，不复制划分逻辑。
import matplotlib.pyplot as plt

split_names = list(dataset_dict.keys())
split_sizes = [len(dataset_dict[name]) for name in split_names]
overlap_counts = [
    [len(split_hashes[row_name] & split_hashes[column_name]) for column_name in split_names]
    for row_name in split_names
]
off_diagonal_overlaps = [
    overlap_counts[row][column]
    for row in range(len(split_names))
    for column in range(len(split_names))
    if row != column
]
diagonal_counts = [overlap_counts[index][index] for index in range(len(split_names))]
if diagonal_counts != split_sizes:
    raise RuntimeError(
        f"Split 内存在重复内容哈希或成员统计不一致：diagonal={diagonal_counts}, sizes={split_sizes}"
    )
if any(off_diagonal_overlaps):
    raise RuntimeError(f"Split 之间存在内容哈希重叠：{overlap_counts}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
bars = axes[0].bar(split_names, split_sizes, color=["#0072B2", "#E69F00", "#009E73"])
axes[0].bar_label(bars)
axes[0].set(title="DatasetDict 实际行数", xlabel="Split", ylabel="记录数")
image = axes[1].imshow(overlap_counts, cmap="Blues", vmin=0)
for row in range(len(split_names)):
    for column in range(len(split_names)):
        axes[1].text(column, row, str(overlap_counts[row][column]), ha="center", va="center")
axes[1].set(
    title="内容哈希交集矩阵", xlabel="Split", ylabel="Split",
    xticks=range(len(split_names)), yticks=range(len(split_names)),
    xticklabels=split_names, yticklabels=split_names,
)
fig.colorbar(image, ax=axes[1], shrink=0.8, label="相同 pair_sha256 数量")
plt.tight_layout()
plt.show()
print({"split_sizes": dict(zip(split_names, split_sizes)), "off_diagonal_overlap": max(off_diagonal_overlaps, default=0)})


非对角线为 0 只证明精确内容哈希没有跨 Split 重复，不能排除同一模板、实体、会话或语义近重复造成的泄漏。本章的模板化小数据仅用于验证数据接口；生产划分仍需保留 `group_id`、来源与时间，并按真实泄漏单位整体分组。


### 5.4．Arrow 持久化与重载

`DatasetDict.save_to_disk()` 保存 Arrow 数据与 Schema，`load_from_disk()` 恢复各 split。本节在临时目录完成保存与重载，使库版本相关的派生产物不进入权威源资产目录。

生产缓存的身份由源数据哈希、处理配置、代码 revision 与依赖版本共同确定。重载验收需要比较 split 名称、Schema、行数、成员顺序和关键内容哈希。


In [ ]:
# 在临时目录演示 Arrow 持久化与重新加载，不提交派生缓存。

from tempfile import TemporaryDirectory


# 使用临时目录隔离中间文件，离开作用域后自动清理。
with TemporaryDirectory(prefix="translation_parallel_hf_") as temporary_directory:
    arrow_path = Path(temporary_directory) / "dataset_dict"
    dataset_dict.save_to_disk(str(arrow_path))
    reloaded_dataset_dict = load_from_disk(str(arrow_path))


    print("DatasetDict Arrow 持久化与重载已完成")


## 6．生产边界

### 6.1．流式读取与物化策略

当前 72 行数据适合使用普通 `Dataset`。当数据规模超过本地磁盘容量，或数据由大量远端 shard 构成时，`streaming=True` 返回 `IterableDataset`，在消费阶段逐条读取。流式数据不提供完整随机访问能力，shuffle 依赖有限缓冲区，跨 worker 与跨设备的分片、断点恢复和成员顺序需要单独设计。

<!-- diagram:dataset-eager-streaming -->
![架构图：常规加载与流式读取的分支架构](assets/figures/20_dataset/dataset-eager-streaming.svg)

[TikZ 源文件](assets/figures/20_dataset/dataset-eager-streaming.tex)

### 6.2．数据资产治理

- `data.jsonl` 是本章的权威源资产；Arrow Cache 与预计算 split 属于可重建派生产物，不与源文件混合管理。
- 当前语料由少量模板组合生成，随机逐行划分只能说明 Datasets 接口，不能提供可信的翻译质量估计。生产数据应保留 `group_id`、来源、许可证和时间信息，并按真实泄漏单位划分。
- Schema 变更、清洗规则变更和划分配置变更均形成新的数据版本；消费者通过版本、哈希和兼容性声明选择资产。
- 文件发布采用临时文件与原子替换；对象存储场景采用不可变对象名和完成标记，避免消费者读取半成品。
- Manifest 至少记录源 URI、revision、许可证、内容哈希、Schema、处理代码 revision、依赖版本、split 成员摘要与恢复入口。
- 流式训练需要显式定义 shard 分配、重试、跳过策略和断点状态，否则恢复后可能重复或遗漏样本。


In [ ]:
# 使用 streaming 模式按需读取样本，避免大数据集一次性进入内存。

streaming_dataset = load_dataset(
    "json",
    data_files=str(DATA_PATH),
    split="train",
    features=translation_features,
    streaming=True,
)
first_two_rows = list(streaming_dataset.take(2))
print(first_two_rows)
